In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [2]:
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
from flax import linen as nn
import optax
import copy

from typing import Callable

from functools import partial

import tensorflow as tf
import tensorrt

from pyDOE import lhs
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import odeint

import numpy as np

from jax.scipy.optimize import minimize
from scipy.optimize import fsolve

import warnings
import math

from collections import deque

2024-12-04 13:36:14.577724: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1733331974.588339    2838 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1733331974.591307    2838 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


W0000 00:00:1733331981.288553    2838 gpu_device.cc:2344] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
print(jax.default_backend())

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


cpu


In [41]:
def flatten_pytree(params):
  values, tree_info = jax.tree_util.tree_flatten(params)

  no_leaves = len(values)
  shapes = []
  flattened = []

  for i in range(no_leaves):
    flattened.append(values[i].flatten())
    shapes.append(values[i].shape)

  return tree_info, shapes, jnp.concatenate(flattened)


def reconstruct_pytree(tree_info, shapes, flattened):

  values = []
  k = 0
  for i in range(len(shapes)):
    values.append(np.reshape(flattened[k:k+np.prod(shapes[i])], shapes[i]))
    k += np.prod(shapes[i])

  py_tree = jax.tree_util.tree_unflatten(tree_info, values)
  return py_tree

In [42]:
t0, tfinal = 0.0, 0.5

In [43]:
def TodaLatt(w, t):
    
    p1, q1, p2, q2, p3, q3 = w

    p1_t = -jnp.exp(q1 - q2) + jnp.exp(q3 - q1)
    q1_t = p1

    p2_t = jnp.exp(q1 - q2) - jnp.exp(q2 - q3)
    q2_t = p2

    p3_t = jnp.exp(q2 - q3) - jnp.exp(q3 - q1)
    q3_t = p3

    dwdt = [p1_t, q1_t, p2_t, q2_t, p3_t, q3_t]

    return dwdt

In [44]:
@jax.jit
def calc_H(p1, p2, p3, q1, q2, q3) :

    H = (p1**2)/2 + (p2**2)/2 + (p3**2)/2 + jnp.exp(q1 - q2) + jnp.exp(q2 - q3) + jnp.exp(q3 - q1)

    return H

In [45]:
@jax.jit
def calc_ab(args) :


    p1, p2, p3, q1, q2, q3 = args

    a1 = -0.5*p1
    a2 = -0.5*p2
    a3 = -0.5*p3

    b1 = 0.5*jnp.exp(0.5*(q1-q2))
    b2 = 0.5*jnp.exp(0.5*(q2-q3))
    b3 = 0.5*jnp.exp(0.5*(q3-q1))

    return a1, a2, a3, b1, b2, b3

In [46]:
@jax.jit
def calc_evals(a1, a2, a3, b1, b2, b3) :
    
    main_diag = jnp.array([a1, a2, a3])
    sub_diag = jnp.array([b1, b2])
    super_diag = jnp.array([b1, b2])
    subsub_diag = jnp.array([b3])
    supersuper_diag = jnp.array([b3])

    L_main = jnp.diag(main_diag)
    L_sub = jnp.diag(sub_diag, k=-1)
    L_super = jnp.diag(super_diag, k=1)
    L_subsub = jnp.diag(subsub_diag, k=-2)
    L_supersuper = jnp.diag(supersuper_diag, k=2)

    L = L_main + L_sub + L_super + L_subsub + L_supersuper

    evals, _ = jnp.linalg.eigh(L)

    return evals

In [47]:
@jax.jit
def calc_evals_pq(args) :

    p1, p2, p3, q1, q2, q3 = args
    a1, a2, a3, b1, b2, b3 = calc_ab((p1, p2, p3, q1, q2, q3))

    evals = jax.vmap(calc_evals, (0, 0, 0, 0, 0, 0), 0)(a1, a2, a3, b1, b2, b3)
    evals = jnp.reshape(evals, (-1,))
    
    return evals
    

In [48]:
def define_collocation_points(t_bdry, N=100):

    ode_points = t_bdry[0] + (t_bdry[1] - t_bdry[0])*lhs(1, N)

    p1s = np.random.uniform(-1.0, 1.0, size=(N, 1))
    q1s = np.random.uniform(-1.0, 1.0, size=(N, 1))
    p2s = np.random.uniform(-1.0, 1.0, size=(N, 1))
    q2s = np.random.uniform(-1.0, 1.0, size=(N, 1))
    p3s = np.random.uniform(-1.0, 1.0, size=(N, 1))
    q3s = np.random.uniform(-1.0, 1.0, size=(N, 1))
    Hs = calc_H(p1s, p2s, p3s, q1s, q2s, q3s)

    evals = jax.vmap(calc_evals_pq, (0), 0)((p1s, p2s, p3s, q1s, q2s, q3s))
    Hs_test = 2*(evals[:, :1]**2 + evals[:, 1:2]**2 + evals[:, 2:]**2)

    ode_points = np.column_stack((ode_points, Hs, evals))
    zsensors = np.column_stack((p1s, q1s, p2s, q2s, p3s, q3s))
    
    return (ode_points, zsensors)

In [49]:
(de_points, zsensors) = define_collocation_points([t0, tfinal], 1000)
print(de_points.shape, zsensors.shape)

(1000, 5) (1000, 6)


In [50]:
n = 500
t = np.linspace(t0, tfinal, n)

p1_init = np.random.uniform(-1.0, 1.0, size=(1,))
q1_init = np.random.uniform(-1.0, 1.0, size=(1,))
p2_init = np.random.uniform(-1.0, 1.0, size=(1,))
q2_init = np.random.uniform(-1.0, 1.0, size=(1,))
p3_init = np.random.uniform(-1.0, 1.0, size=(1,))
q3_init = np.random.uniform(-1.0, 1.0, size=(1,))
H_init = calc_H(p1_init, p2_init, p3_init, q1_init, q2_init, q3_init)

data_rk = np.column_stack([p1_init, q1_init, p2_init, q2_init, p3_init, q3_init])
data_rk = np.reshape(data_rk, (-1,))
t_rk = t

sol_rk = odeint(TodaLatt, data_rk, t_rk, rtol = 1e-13, atol = 1e-13)
p1_rk, q1_rk, p2_rk, q2_rk, p3_rk, q3_rk = sol_rk[:, 0], sol_rk[:, 1], sol_rk[:, 2], sol_rk[:, 3], sol_rk[:, 4], sol_rk[:, 5]
H_rk = calc_H(p1_rk, p2_rk, p3_rk, q1_rk, q2_rk, q3_rk)

In [51]:
class Normalize(nn.Module):
  xmin: float
  xmax: float

  @nn.compact
  def __call__(self, x):
    return 2.0*(x-self.xmin)/(self.xmax - self.xmin) - 1.0

In [52]:
class HardConstraint(nn.Module):
  t0: float
  tfinal: float
  u0: float

  @nn.compact
  def __call__(self, inputs):
    t, nn = inputs
    return self.u0 + (t-self.t0)/(self.tfinal-self.t0)*nn

In [53]:
class CombineBranches(nn.Module):
  @nn.compact
  def __call__(self, inp1, inp2):
    mult = jnp.sum(inp1*inp2, axis=1)
    out = jnp.reshape(mult, (-1,1))
    return out

In [54]:
class MLP(nn.Module):

  layers: int
  units: int

  @nn.compact
  def __call__(self, inp):
    b = inp
    for i in range(self.layers-1):
      b = nn.Dense(self.units)(b)
      b = nn.tanh(b)
    out = nn.Dense(6*self.units)(b)
    return out

In [55]:
class DeepONet(nn.Module):

  t0: float
  tfinal: float
  layers: int 
  units: int

  @nn.compact
  def __call__(self, t, u):

    if u.ndim == 1:
      u = jnp.reshape(u, (1,-1))

    if t.ndim == 1:
      t = jnp.reshape(t, (1,-1)) 

    t_0 = t[:, :1]
    H_0 = t[:, 1:2]
    eval1_0 = t[:, 2:3]
    eval2_0 = t[:, 3:4]
    eval3_0 = t[:, 4:]
      
    t_0 = jnp.reshape(t_0, (-1,1))
    t_0_norm = Normalize(self.t0, self.tfinal)(t_0)
   
    p1_0 = u[:, :1]
    q1_0 = u[:, 1:2]
    p2_0 = u[:, 2:3]
    q2_0 = u[:, 3:4]
    p3_0 = u[:, 4:5]
    q3_0 = u[:, 5:]

    u = jnp.column_stack((p1_0, q1_0, p2_0, q2_0, p3_0, q3_0))

    b = t_0_norm
    # b = jnp.column_stack((t_0_norm, H_0))

    trunk_net = MLP(self.layers, self.units)(b)
    branch_net = MLP(self.layers, self.units)(u)

    trunk_p1 = trunk_net[:, :self.units]
    trunk_q1 = trunk_net[:, self.units:2*self.units]
    trunk_p2 = trunk_net[:, 2*self.units:3*self.units]
    trunk_q2 = trunk_net[:, 3*self.units:4*self.units]
    trunk_p3 = trunk_net[:, 4*self.units:5*self.units]
    trunk_q3 = trunk_net[:, 5*self.units:]
      
    branch_p1 = branch_net[:, :self.units]
    branch_q1 = branch_net[:, self.units:2*self.units]
    branch_p2 = branch_net[:, 2*self.units:3*self.units]
    branch_q2 = branch_net[:, 3*self.units:4*self.units]
    branch_p3 = branch_net[:, 4*self.units:5*self.units]
    branch_q3 = branch_net[:, 5*self.units:]
      
    p1 = CombineBranches()(trunk_p1, branch_p1)
    q1 = CombineBranches()(trunk_q1, branch_q1)
    p2 = CombineBranches()(trunk_p2, branch_p2)
    q2 = CombineBranches()(trunk_q2, branch_q2)
    p3 = CombineBranches()(trunk_p3, branch_p3)
    q3 = CombineBranches()(trunk_q3, branch_q3)
      
    p1 = HardConstraint(self.t0, self.tfinal, p1_0)([t_0, p1])
    q1 = HardConstraint(self.t0, self.tfinal, q1_0)([t_0, q1])
    p2 = HardConstraint(self.t0, self.tfinal, p2_0)([t_0, p2])
    q2 = HardConstraint(self.t0, self.tfinal, q2_0)([t_0, q2])
    p3 = HardConstraint(self.t0, self.tfinal, p3_0)([t_0, p3])
    q3 = HardConstraint(self.t0, self.tfinal, q3_0)([t_0, q3])
      
    p1 = jnp.reshape(p1, (-1,))
    q1 = jnp.reshape(q1, (-1,))
    p2 = jnp.reshape(p2, (-1,))
    q2 = jnp.reshape(q2, (-1,))
    p3 = jnp.reshape(p3, (-1,))
    q3 = jnp.reshape(q3, (-1,))

    return p1, q1, p2, q2, p3, q3

In [56]:
@partial(jax.jit, static_argnums=(2,))
def var_model(t, z, component, params):
  return deeponet.apply(params, t, z)[component][0]

@partial(jax.jit, static_argnums=(2,))
def var(t, z, component, params):
  return jax.vmap(var_model, [0, 0, None, None])(t, z, component, params)

@partial(jax.jit, static_argnums=(2,))
def var_t(t, z, component, params):
  return jax.vmap(jax.grad(var_model, 0), [0, 0, None, None])(t, z, component, params)

@jax.jit
def loss(params, t, z):
    
    p1_t_loss_w = 1.0
    q1_t_loss_w = 1.0
    p2_t_loss_w = 1.0
    q2_t_loss_w = 1.0
    p3_t_loss_w = 1.0
    q3_t_loss_w = 1.0    
    
    diff_loss_w = 1.0
    H_loss_w = 1.0

    t_0 = t[:, :1]
    H_0 = t[:, 1:2]
    # eval1_0 = t[:, 2:3]
    # eval2_0 = t[:, 3:4]
    # eval3_0 = t[:, 4:]
    
    p1_0 = z[:, :1]
    q1_0 = z[:, 1:2]
    p2_0 = z[:, 2:3]
    q2_0 = z[:, 3:4]
    p3_0 = z[:, 4:5]
    q3_0 = z[:, 5:]
    
    p1 = var(t, z, 0, params)
    q1 = var(t, z, 1, params)
    p2 = var(t, z, 2, params)
    q2 = var(t, z, 3, params)
    p3 = var(t, z, 4, params)
    q3 = var(t, z, 5, params)
    
    H = calc_H(p1, p2, p3, q1, q2, q3)
    H = jnp.reshape(H, (-1, 1))
    
    dp1_t = var_t(t, z, 0, params)
    dq1_t = var_t(t, z, 1, params)
    dp2_t = var_t(t, z, 2, params)
    dq2_t = var_t(t, z, 3, params)
    dp3_t = var_t(t, z, 4, params)
    dq3_t = var_t(t, z, 5, params)
        
    p1_t = dp1_t[:, 0]
    q1_t = dq1_t[:, 0]  
    p2_t = dp2_t[:, 0]
    q2_t = dq2_t[:, 0]
    p3_t = dp3_t[:, 0]
    q3_t = dq3_t[:, 0]
    
    H_loss = H - H_0
    H_loss = jnp.square(H_loss)
    H_loss = jnp.mean(H_loss)

    p1_t_loss = p1_t - (-jnp.exp(q1 - q2) + jnp.exp(q3 - q1))
    p1_t_loss = jnp.square(p1_t_loss)
    p1_t_loss = jnp.mean(p1_t_loss)

    q1_t_loss = q1_t - p1
    q1_t_loss = jnp.square(q1_t_loss)
    q1_t_loss = jnp.mean(q1_t_loss)
    
    p2_t_loss = p2_t - (jnp.exp(q1 - q2) - jnp.exp(q2 - q3))
    p2_t_loss = jnp.square(p2_t_loss)
    p2_t_loss = jnp.mean(p2_t_loss)
    
    q2_t_loss = q2_t - p2
    q2_t_loss = jnp.square(q2_t_loss)
    q2_t_loss = jnp.mean(q2_t_loss)

    p3_t_loss = p3_t - (jnp.exp(q2 - q3) - jnp.exp(q3 - q1))
    p3_t_loss = jnp.square(p3_t_loss)
    p3_t_loss = jnp.mean(p3_t_loss)
    
    q3_t_loss = q3_t - p3
    q3_t_loss = jnp.square(q3_t_loss)
    q3_t_loss = jnp.mean(q3_t_loss)
        
    diff_loss = p1_t_loss_w*p1_t_loss +  q1_t_loss_w*q1_t_loss + p2_t_loss_w*p2_t_loss + q2_t_loss_w*q2_t_loss + \
                p3_t_loss_w*p3_t_loss + q3_t_loss_w*q3_t_loss

    total_loss = diff_loss_w*diff_loss + H_loss_w*H_loss

    return total_loss

@jax.jit
def train_step(params, pdes, z):
    t = pdes[:, :]
    
    return jax.value_and_grad(loss, has_aux = False)(params, t, z)

@partial(jax.jit, static_argnums=(3,))
def optimize(grads, opt_state, params, optimizer_update):
    updates, opt_state = optimizer_update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state

In [57]:
def train_network(params, des, sensors, epochs=100):

    N = de_points.shape[0]
    
    nr_batches = 10
    batch_size = len(des)//nr_batches
    
    print("batch size:", batch_size)
    
    lr = 1e-3
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(params)
    
    ds_z = tf.data.Dataset.from_tensor_slices(sensors)
    ds_de = tf.data.Dataset.from_tensor_slices(des)
    
    ds = tf.data.Dataset.zip((ds_de, ds_z))
    ds = ds.shuffle(N).batch(batch_size)
    
    epoch_loss = np.zeros(epochs)

    for i in range(epochs):

        for (des_batch, z_batch) in ds:

            loss, grads = train_step(params, des_batch.numpy(), z_batch.numpy())
            params, opt_state = optimize(grads, opt_state, params, optimizer.update)

            epoch_loss[i] += loss

        epoch_loss[i] /= batch_size

        if i % 100  == 0:
            print(f'Loss in epoch {i}: {epoch_loss[i]}')

    return params, epoch_loss

In [58]:
deeponet = DeepONet(t0, tfinal, layers=4, units=40)

t_init = jnp.full((10, 5), 1.0)
u_init = jnp.full((10, 6), 1.0)

params = deeponet.init(jax.random.PRNGKey(0), t_init, u_init)

In [59]:
param_count = sum(x.size for x in jax.tree.leaves(params))
print(param_count)

26600


In [60]:
# epochs = 5
# params, loss = train_network(params, de_points, zsensors, epochs)

# plt.semilogy(loss)
# plt.grid()

In [61]:
# tree_info, shapes, params_flat = flatten_pytree(params)
# jnp.save("P_t0.5_1000_1000_ep_Toda3", params_flat)

tree_info, shapes, _ = flatten_pytree(params)
params_flat = jnp.load("P_t0.5_1000_5000_ep_Toda3.npy")
params = reconstruct_pytree(tree_info, shapes, params_flat)

In [62]:
class EDeepONet(nn.Module):

  t0: float
  tfinal: float
  layers: int
  units: int

  @nn.compact
  def __call__(self, t, u):

    if u.ndim == 1:
      u = jnp.reshape(u, (1,-1))

    if t.ndim == 1:
      t = jnp.reshape(t, (1,-1)) 

    t_0 = t[:, :1]
    H_0 = t[:, 1:2]
    eval1_0 = t[:, 2:3]
    eval2_0 = t[:, 3:4]
    eval3_0 = t[:, 4:]
      
    t_0 = jnp.reshape(t_0, (-1,1))
    t_0_norm = Normalize(self.t0, self.tfinal)(t_0)
   
    p1_0 = u[:, :1]
    q1_0 = u[:, 1:2]
    p2_0 = u[:, 2:3]
    q2_0 = u[:, 3:4]
    p3_0 = u[:, 4:5]
    q3_0 = u[:, 5:]

    u = jnp.column_stack((p1_0, q1_0, p2_0, q2_0, p3_0, q3_0))

    # b = t_0_norm
    # b = jnp.column_stack((t_0_norm, H_0))
    b = jnp.column_stack((t_0_norm, eval1_0, eval2_0, eval3_0))

    trunk_net = MLP(self.layers, self.units)(b)
    branch_net = MLP(self.layers, self.units)(u)

    trunk_p1 = trunk_net[:, :self.units]
    trunk_q1 = trunk_net[:, self.units:2*self.units]
    trunk_p2 = trunk_net[:, 2*self.units:3*self.units]
    trunk_q2 = trunk_net[:, 3*self.units:4*self.units]
    trunk_p3 = trunk_net[:, 4*self.units:5*self.units]
    trunk_q3 = trunk_net[:, 5*self.units:]
      
    branch_p1 = branch_net[:, :self.units]
    branch_q1 = branch_net[:, self.units:2*self.units]
    branch_p2 = branch_net[:, 2*self.units:3*self.units]
    branch_q2 = branch_net[:, 3*self.units:4*self.units]
    branch_p3 = branch_net[:, 4*self.units:5*self.units]
    branch_q3 = branch_net[:, 5*self.units:]
      
    p1 = CombineBranches()(trunk_p1, branch_p1)
    q1 = CombineBranches()(trunk_q1, branch_q1)
    p2 = CombineBranches()(trunk_p2, branch_p2)
    q2 = CombineBranches()(trunk_q2, branch_q2)
    p3 = CombineBranches()(trunk_p3, branch_p3)
    q3 = CombineBranches()(trunk_q3, branch_q3)
      
    p1 = HardConstraint(self.t0, self.tfinal, p1_0)([t_0, p1])
    q1 = HardConstraint(self.t0, self.tfinal, q1_0)([t_0, q1])
    p2 = HardConstraint(self.t0, self.tfinal, p2_0)([t_0, p2])
    q2 = HardConstraint(self.t0, self.tfinal, q2_0)([t_0, q2])
    p3 = HardConstraint(self.t0, self.tfinal, p3_0)([t_0, p3])
    q3 = HardConstraint(self.t0, self.tfinal, q3_0)([t_0, q3])
      
    p1 = jnp.reshape(p1, (-1,))
    q1 = jnp.reshape(q1, (-1,))
    p2 = jnp.reshape(p2, (-1,))
    q2 = jnp.reshape(q2, (-1,))
    p3 = jnp.reshape(p3, (-1,))
    q3 = jnp.reshape(q3, (-1,))

    # evals = jax.vmap(calc_evals_pq, [0])((p1, p2, p3, q1, q2, q3))

    # eval1 = evals[:, :1]
    # eval2 = evals[:, 1:2]
    # eval3 = evals[:, 2:]

    # eval1 = jnp.reshape(eval1, (-1,))
    # eval2 = jnp.reshape(eval2, (-1,))
    # eval3 = jnp.reshape(eval3, (-1,))      

    return p1, q1, p2, q2, p3, q3

In [63]:
@partial(jax.jit, static_argnums=(2,))
def Evar_model(t, z, component, params):
  return Edeeponet.apply(params, t, z)[component][0]

@partial(jax.jit, static_argnums=(2,))
def Evar(t, z, component, params):
  return jax.vmap(Evar_model, [0, 0, None, None])(t, z, component, params)

@partial(jax.jit, static_argnums=(2,))
def Evar_t(t, z, component, params):
  return jax.vmap(jax.grad(Evar_model, 0), [0, 0, None, None])(t, z, component, params)       
    
@jax.jit
def Eloss(params, t, z):
    
    p1_t_loss_w = 1.0
    q1_t_loss_w = 1.0
    p2_t_loss_w = 1.0
    q2_t_loss_w = 1.0
    p3_t_loss_w = 1.0
    q3_t_loss_w = 1.0    
    
    diff_loss_w = 1.0
    H_loss_w = 1.0

    t_0 = t[:, :1]
    H_0 = t[:, 1]
    
    p1_0 = z[:, :1]
    q1_0 = z[:, 1:2]
    p2_0 = z[:, 2:3]
    q2_0 = z[:, 3:4]
    p3_0 = z[:, 4:5]
    q3_0 = z[:, 5:]

    p1 = Evar(t, z, 0, params)
    q1 = Evar(t, z, 1, params)
    p2 = Evar(t, z, 2, params)
    q2 = Evar(t, z, 3, params)
    p3 = Evar(t, z, 4, params)
    q3 = Evar(t, z, 5, params)
    
    H = calc_H(p1, p2, p3, q1, q2, q3)
    
    dp1_t = Evar_t(t, z, 0, params)
    dq1_t = Evar_t(t, z, 1, params)
    dp2_t = Evar_t(t, z, 2, params)
    dq2_t = Evar_t(t, z, 3, params)
    dp3_t = Evar_t(t, z, 4, params)
    dq3_t = Evar_t(t, z, 5, params)
        
    p1_t = dp1_t[:, 0]
    q1_t = dq1_t[:, 0]  
    p2_t = dp2_t[:, 0]
    q2_t = dq2_t[:, 0]
    p3_t = dp3_t[:, 0]
    q3_t = dq3_t[:, 0]

    H_loss = H - H_0
    H_loss = jnp.square(H_loss)
    H_loss = jnp.mean(H_loss)

    p1_t_loss = p1_t - (-jnp.exp(q1 - q2) + jnp.exp(q3 - q1))
    p1_t_loss = jnp.square(p1_t_loss)
    p1_t_loss = jnp.mean(p1_t_loss)

    q1_t_loss = q1_t - p1
    q1_t_loss = jnp.square(q1_t_loss)
    q1_t_loss = jnp.mean(q1_t_loss)
    
    p2_t_loss = p2_t - (jnp.exp(q1 - q2) - jnp.exp(q2 - q3))
    p2_t_loss = jnp.square(p2_t_loss)
    p2_t_loss = jnp.mean(p2_t_loss)
    
    q2_t_loss = q2_t - p2
    q2_t_loss = jnp.square(q2_t_loss)
    q2_t_loss = jnp.mean(q2_t_loss)

    p3_t_loss = p3_t - (jnp.exp(q2 - q3) - jnp.exp(q3 - q1))
    p3_t_loss = jnp.square(p3_t_loss)
    p3_t_loss = jnp.mean(p3_t_loss)
    
    q3_t_loss = q3_t - p3
    q3_t_loss = jnp.square(q3_t_loss)
    q3_t_loss = jnp.mean(q3_t_loss)
        
    diff_loss = p1_t_loss_w*p1_t_loss +  q1_t_loss_w*q1_t_loss + p2_t_loss_w*p2_t_loss + q2_t_loss_w*q2_t_loss + \
                p3_t_loss_w*p3_t_loss + q3_t_loss_w*q3_t_loss

    total_loss = diff_loss_w*diff_loss + H_loss_w*H_loss
    
    return total_loss, H_loss

@jax.jit
def Etrain_step(params, pdes, z):
    t = pdes[:, :]
    
    return jax.value_and_grad(Eloss, has_aux = True)(params, t, z)

@partial(jax.jit, static_argnums=(3,))
def Eoptimize(grads, opt_state, params, optimizer_update):
    updates, opt_state = optimizer_update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state

In [64]:
def Etrain_network(params, des, sensors, epochs=100):

    N = de_points.shape[0]
    
    nr_batches = 10
    batch_size = len(des)//nr_batches
    
    print("batch size:", batch_size)
    
    lr = 1e-3
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(params)
    
    ds_z = tf.data.Dataset.from_tensor_slices(sensors)
    ds_de = tf.data.Dataset.from_tensor_slices(des)
    
    ds = tf.data.Dataset.zip((ds_de, ds_z))
    ds = ds.shuffle(N).batch(batch_size)
    
    epoch_loss = np.zeros(epochs)
    epoch_aux_loss = np.zeros(epochs)

    for i in range(epochs):

        for (des_batch, z_batch) in ds:

            losses, grads = Etrain_step(params, des_batch.numpy(), z_batch.numpy())
            params, opt_state = Eoptimize(grads, opt_state, params, optimizer.update)

            loss, aux_loss = losses
           
            epoch_loss[i] += loss
            epoch_aux_loss[i] += aux_loss

        epoch_loss[i] /= batch_size
        epoch_aux_loss[i] /= batch_size

        if i % 100  == 0:
            print(f'Loss in epoch {i}: {epoch_loss[i]}')
            print(f'Aux Loss in epoch {i}: {epoch_aux_loss[i]}')

    return params, epoch_loss

In [65]:
Edeeponet = EDeepONet(t0, tfinal, layers=4, units=40)

t_init = jnp.full((10, 5), 1.0)
u_init = jnp.full((10, 6), 1.0)

Eparams = Edeeponet.init(jax.random.PRNGKey(0), t_init, u_init)

In [66]:
Eparam_count = sum(x.size for x in jax.tree.leaves(Eparams))
print(Eparam_count)

26720


In [67]:
# epochs = 5
# Eparams, Eloss = Etrain_network(Eparams, de_points, zsensors, epochs)

# plt.semilogy(Eloss)
# plt.grid()

In [68]:
# tree_info, shapes, Eparams_flat = flatten_pytree(Eparams)
# jnp.save("E_t0.5_1000_1000_ep_Toda3", Eparams_flat)

tree_info, shapes, _ = flatten_pytree(Eparams)
Eparams_flat = jnp.load("E_t0.5_1000_5000_ep_Toda3.npy")
Eparams = reconstruct_pytree(tree_info, shapes, Eparams_flat)

In [69]:
# @jax.jit
# def loss_fun(args, e_0) :

#     args = jnp.reshape(args, (-1,))
#     e_0 = jnp.reshape(e_0, (-1,))

#     p1, p2, p3, q1, q2, q3 = args[0], args[1], args[2], args[3], args[4], args[5]
#     e1_0, e2_0, e3_0 = e_0[0], e_0[1], e_0[2]

#     a1, a2, a3, b1, b2, b3 = calc_ab(args)

#     args_new = (a1, a2, a3, b1, b2, b3)

#     evals = calc_evals(a1, a2, a3, b1, b2, b3)
    
#     e1 = evals[0]
#     e2 = evals[1]
#     e3 = evals[2]

#     f1 = (e1 - e1_0)**2
#     f2 = (e2 - e2_0)**2
#     f3 = (e3 - e3_0)**2

#     f_arr = jnp.array([[f1], [f2], [f3]])
#     norm_f = jnp.linalg.norm(f_arr, ord = jnp.inf)

#     # f = f1 + f2 + f3

#     return norm_f
#     #return jnp.array([[f]])


# @jax.jit
# def get_derivs(args, e_0, e) :

#     a1, a2, a3, b1, b2, b3 = args[0], args[1], args[2], args[3], args[4], args[5]

#     dF_e = 3*e**2 - 2*(a1 + a2 + a3)*e + (a1*a2 + a2*a3 + a1*a3 - b1**2 - b2**2 - b3**2)
#     dF_a1 = -e**2 + (a2 + a3)*e - (a2*a3 - b2**2)
#     dF_a2 = -e**2 + (a1 + a3)*e - (a1*a3 - b3**2)
#     dF_a3 = -e**2 + (a1 + a2)*e - (a1*a2 - b1**2)
#     dF_b1 = -2*b1*e - (-2*a3*b1 + 2*b2*b3)
#     dF_b2 = -2*b2*e - (-2*a1*b2 + 2*b1*b3)
#     dF_b3 = -2*b3*e - (-2*a2*b3 + 2*b1*b2)

#     de_a1 = -dF_a1/dF_e
#     de_a2 = -dF_a2/dF_e
#     de_a3 = -dF_a3/dF_e
#     de_b1 = -dF_b1/dF_e
#     de_b2 = -dF_b2/dF_e
#     de_b3 = -dF_b3/dF_e

#     df_a1 = 2*(e - e_0)*de_a1
#     df_a2 = 2*(e - e_0)*de_a2
#     df_a3 = 2*(e - e_0)*de_a3
#     df_b1 = 2*(e - e_0)*de_b1
#     df_b2 = 2*(e - e_0)*de_b2
#     df_b3 = 2*(e - e_0)*de_b3

#     df_x = (df_a1, df_a2, df_a3, df_b1, df_b2, df_b3)

#     return df_x


# # @jax.jit
# # def grad_loss_fun(args, e_0) :

# #     args = jnp.reshape(args, (-1,))
# #     e_0 = jnp.reshape(e_0, (-1,))

# #     p1, p2, p3, q1, q2, q3 = args[0], args[1], args[2], args[3], args[4], args[5]
# #     e1_0, e2_0, e3_0 = e_0[0], e_0[1], e_0[2]

# #     a1, a2, a3, b1, b2, b3 = calc_ab(args)
    
# #     args_new = (a1, a2, a3, b1, b2, b3)

# #     evals = calc_evals(a1, a2, a3, b1, b2, b3)
    
# #     e1 = evals[0]
# #     e2 = evals[1]
# #     e3 = evals[2]

# #     df1_a1, df1_a2, df1_a3, df1_b1, df1_b2, df1_b3 = get_derivs(args_new, e1_0, e1)
# #     df2_a1, df2_a2, df2_a3, df2_b1, df2_b2, df2_b3 = get_derivs(args_new, e2_0, e2)
# #     df3_a1, df3_a2, df3_a3, df3_b1, df3_b2, df3_b3 = get_derivs(args_new, e3_0, e3)

# #     df1_p1, df2_p1, df3_p1 = df1_a1*-0.5, df2_a1*-0.5, df3_a1*-0.5
# #     df1_p2, df2_p2, df3_p2 = df1_a2*-0.5, df2_a2*-0.5, df3_a2*-0.5
# #     df1_p3, df2_p3, df3_p3 = df1_a3*-0.5, df2_a3*-0.5, df3_a3*-0.5

# #     df1_q1 = df1_b1*0.5*b1 + df1_b3*-0.5*b3
# #     df2_q1 = df2_b1*0.5*b1 + df2_b3*-0.5*b3
# #     df3_q1 = df3_b1*0.5*b1 + df3_b3*-0.5*b3

# #     df1_q2 = df1_b1*-0.5*b1 + df1_b2*0.5*b2
# #     df2_q2 = df2_b1*-0.5*b1 + df2_b2*0.5*b2
# #     df3_q2 = df3_b1*-0.5*b1 + df3_b2*0.5*b2

# #     df1_q3 = df1_b2*-0.5*b2 + df1_b3*0.5*b3
# #     df2_q3 = df2_b2*-0.5*b2 + df2_b3*0.5*b3
# #     df3_q3 = df3_b2*-0.5*b2 + df3_b3*0.5*b3

# #     df1 = (df1_p1, df1_p2, df1_p3, df1_q1, df1_q2, df1_q3)
# #     df2 = (df2_p1, df2_p2, df2_p3, df2_q1, df2_q2, df2_q3)
# #     df3 = (df3_p1, df3_p2, df3_p3, df3_q1, df3_q2, df3_q3)

# #     df_p1 = df1_p1 + df2_p1 + df3_p1
# #     df_p2 = df1_p2 + df2_p2 + df3_p2
# #     df_p3 = df1_p3 + df2_p3 + df3_p3
# #     df_q1 = df1_q1 + df2_q1 + df3_q1
# #     df_q2 = df1_q2 + df2_q2 + df3_q2
# #     df_q3 = df1_q3 + df2_q3 + df3_q3

# #     grad_f = jnp.array([[df_p1], [df_p2], [df_p3], [df_q1], [df_q2], [df_q3]])

# #     return grad_f

In [70]:
# @jax.jit
# def grad_loss_fun(args, e_0) :

#     return jax.grad(loss_fun)(args, e_0)

In [71]:
w_H = 0.5
w_C = 0.5

@jax.jit
def loss_fun(args, e_0) :

    args = jnp.reshape(args, (-1,))
    e_0 = jnp.reshape(e_0, (-1,))

    p1, p2, p3, q1, q2, q3 = args[0], args[1], args[2], args[3], args[4], args[5]
    e1_0, e2_0, e3_0 = e_0[0], e_0[1], e_0[2]
    
    H_0 = 2*(e1_0**2 + e2_0**2 + e3_0**2)
    H = calc_H(p1, p2, p3, q1, q2, q3)

    fH = (H - H_0)**2

    a1, a2, a3, b1, b2, b3 = calc_ab(args)
    args_new = (a1, a2, a3, b1, b2, b3)
    evals = calc_evals(a1, a2, a3, b1, b2, b3)
    
    e1 = evals[0]
    e2 = evals[1]
    e3 = evals[2]

    f1 = (e1 - e1_0)**2
    f2 = (e2 - e2_0)**2
    f3 = (e3 - e3_0)**2

    fC = f1 + f2 + f3

    f = w_H*fH + w_c*fC

    # return f
    return jnp.array([[f]])


@jax.jit
def get_derivs(args, e_0, e) :

    a1, a2, a3, b1, b2, b3 = args[0], args[1], args[2], args[3], args[4], args[5]

    dF_e = 3*e**2 - 2*(a1 + a2 + a3)*e + (a1*a2 + a2*a3 + a1*a3 - b1**2 - b2**2 - b3**2)
    dF_a1 = -e**2 + (a2 + a3)*e - (a2*a3 - b2**2)
    dF_a2 = -e**2 + (a1 + a3)*e - (a1*a3 - b3**2)
    dF_a3 = -e**2 + (a1 + a2)*e - (a1*a2 - b1**2)
    dF_b1 = -2*b1*e - (-2*a3*b1 + 2*b2*b3)
    dF_b2 = -2*b2*e - (-2*a1*b2 + 2*b1*b3)
    dF_b3 = -2*b3*e - (-2*a2*b3 + 2*b1*b2)

    de_a1 = -dF_a1/dF_e
    de_a2 = -dF_a2/dF_e
    de_a3 = -dF_a3/dF_e
    de_b1 = -dF_b1/dF_e
    de_b2 = -dF_b2/dF_e
    de_b3 = -dF_b3/dF_e

    df_a1 = 2*(e - e_0)*de_a1
    df_a2 = 2*(e - e_0)*de_a2
    df_a3 = 2*(e - e_0)*de_a3
    df_b1 = 2*(e - e_0)*de_b1
    df_b2 = 2*(e - e_0)*de_b2
    df_b3 = 2*(e - e_0)*de_b3

    df_x = (df_a1, df_a2, df_a3, df_b1, df_b2, df_b3)

    return df_x


@jax.jit
def grad_loss_fun(args, e_0) :

    args = jnp.reshape(args, (-1,))
    e_0 = jnp.reshape(e_0, (-1,))

    p1, p2, p3, q1, q2, q3 = args[0], args[1], args[2], args[3], args[4], args[5]
    e1_0, e2_0, e3_0 = e_0[0], e_0[1], e_0[2]

    a1, a2, a3, b1, b2, b3 = calc_ab(args)
    args_new = (a1, a2, a3, b1, b2, b3)
    
    evals = calc_evals(a1, a2, a3, b1, b2, b3)
    
    e1 = evals[0]
    e2 = evals[1]
    e3 = evals[2]

    H_0 = 2*(e1_0**2 + e2_0**2 + e3_0**2)
    H = calc_H(p1, p2, p3, q1, q2, q3)

    dH_p1 = p1
    dH_p2 = p2
    dH_p3 = p3
    dH_q1 = jnp.exp(q1 - q2) - jnp.exp(q3 - q1)
    dH_q2 = -jnp.exp(q1 - q2) + jnp.exp(q2 - q3)
    dH_q3 = -jnp.exp(q2 - q3) + jnp.exp(q3 - q1)

    dfH_p1 = 2*(H - H_0)*dH_p1
    dfH_p2 = 2*(H - H_0)*dH_p2
    dfH_p3 = 2*(H - H_0)*dH_p3
    dfH_q1 = 2*(H - H_0)*dH_q1
    dfH_q2 = 2*(H - H_0)*dH_q2
    dfH_q3 = 2*(H - H_0)*dH_q3

    grad_fH = jnp.array([[dfH_p1], [dfH_p2], [dfH_p3], [dfH_q1], [dfH_q2], [dfH_q3]])

    df1_a1, df1_a2, df1_a3, df1_b1, df1_b2, df1_b3 = get_derivs(args_new, e1_0, e1)
    df2_a1, df2_a2, df2_a3, df2_b1, df2_b2, df2_b3 = get_derivs(args_new, e2_0, e2)
    df3_a1, df3_a2, df3_a3, df3_b1, df3_b2, df3_b3 = get_derivs(args_new, e3_0, e3)

    df1_p1, df2_p1, df3_p1 = df1_a1*-0.5, df2_a1*-0.5, df3_a1*-0.5
    df1_p2, df2_p2, df3_p2 = df1_a2*-0.5, df2_a2*-0.5, df3_a2*-0.5
    df1_p3, df2_p3, df3_p3 = df1_a3*-0.5, df2_a3*-0.5, df3_a3*-0.5

    df1_q1 = df1_b1*0.5*b1 + df1_b3*-0.5*b3
    df2_q1 = df2_b1*0.5*b1 + df2_b3*-0.5*b3
    df3_q1 = df3_b1*0.5*b1 + df3_b3*-0.5*b3

    df1_q2 = df1_b1*-0.5*b1 + df1_b2*0.5*b2
    df2_q2 = df2_b1*-0.5*b1 + df2_b2*0.5*b2
    df3_q2 = df3_b1*-0.5*b1 + df3_b2*0.5*b2

    df1_q3 = df1_b2*-0.5*b2 + df1_b3*0.5*b3
    df2_q3 = df2_b2*-0.5*b2 + df2_b3*0.5*b3
    df3_q3 = df3_b2*-0.5*b2 + df3_b3*0.5*b3

    df1 = (df1_p1, df1_p2, df1_p3, df1_q1, df1_q2, df1_q3)
    df2 = (df2_p1, df2_p2, df2_p3, df2_q1, df2_q2, df2_q3)
    df3 = (df3_p1, df3_p2, df3_p3, df3_q1, df3_q2, df3_q3)

    dfC_p1 = df1_p1 + df2_p1 + df3_p1
    dfC_p2 = df1_p2 + df2_p2 + df3_p2
    dfC_p3 = df1_p3 + df2_p3 + df3_p3
    dfC_q1 = df1_q1 + df2_q1 + df3_q1
    dfC_q2 = df1_q2 + df2_q2 + df3_q2
    dfC_q3 = df1_q3 + df2_q3 + df3_q3

    grad_fC = jnp.array([[dfC_p1], [dfC_p2], [dfC_p3], [dfC_q1], [dfC_q2], [dfC_q3]])

    return w_H*grad_fH + w_c*grad_fC

In [72]:
test_args = jnp.array([1., 2., 3., 4., 5., 6.])
test_e0 = jnp.array([1., 2., 3.])

print(grad_loss_fun(test_args, test_e0))

NameError: name 'w_c' is not defined

In [ ]:
@jax.jit
def find_rec(x):
    
    x = x + jnp.finfo(jnp.float64).eps
    
    return 1/x


@jax.jit
def wolfe_conds(args):

    alpha, curr_x, curr_f, curr_grad, next_x, next_f, next_grad, p, e_0, iter = args

    c1 = 1e-3

    cond = next_f - curr_f - c1*alpha*(curr_grad.T@p)

    cond_bool = jax.lax.cond(cond[0][0] > 0.0 , lambda x: True, lambda x: False, None)
    final_bool = cond_bool
    
    return final_bool


@jax.jit
def update_alpha(args) :

    alpha, curr_x, curr_f, curr_grad, next_x, next_f, next_grad, p, e_0, iter = args

    alpha = alpha*0.1
    iter = iter + 1
        
    next_x = curr_x + alpha*p
    next_f = loss_fun(next_x, e_0)
    next_grad = grad_loss_fun(next_x, e_0) 
   
    return (alpha, curr_x, curr_f, curr_grad, next_x, next_f, next_grad, p, e_0, iter)


@jax.jit
def BFGS(x, e_0) :

    iter_num = 0
    init_iter = 0
    alpha = 1.0
    I_3 = jnp.identity(6, dtype = jnp.float64)
    B_0_inv = jnp.identity(6, dtype = jnp.float64)

    e_0 = jnp.reshape(e_0, (-1, 1))
    
    x_0 = x
    x_0 = jnp.reshape(x_0, (-1, 1))

    D_0 = loss_fun(x_0, e_0)
    gradD_0 = grad_loss_fun(x_0, e_0)

    p = -(B_0_inv@gradD_0)

    pot_x = x_0 + alpha*p
    pot_D = loss_fun(pot_x, e_0)
    pot_gradD = grad_loss_fun(pot_x, e_0)

    init_args = (alpha, x_0, D_0, gradD_0, pot_x, pot_D, pot_gradD, p, e_0, init_iter)
    
    step, _, _, _, _, _, _, _, _, _ = jax.lax.while_loop(wolfe_conds, update_alpha, init_args)

    x_1 = x_0 + step*p

    while iter_num < 2:

        delta_x = x_1 - x_0

        D_1 = loss_fun(x_1, e_0)
        gradD_1 = grad_loss_fun(x_1, e_0)

        y = gradD_1 - gradD_0       

        y_T_delta_x = y.T@delta_x
        y_T_delta_x_rec = find_rec(y_T_delta_x)
        
        B_1_inv = ((I_3 - (y_T_delta_x_rec*(delta_x@y.T)))@B_0_inv@(I_3 - (y_T_delta_x_rec*(y@delta_x.T)))) + (y_T_delta_x_rec*(delta_x@delta_x.T))   
        p = -(B_1_inv@gradD_1)
        
        pot_x = x_1 + alpha*p
        pot_D = loss_fun(pot_x, e_0)
        pot_gradD = grad_loss_fun(pot_x, e_0) 
                  
        init_args = (alpha, x_1, D_1, gradD_1, pot_x, pot_D, pot_gradD, p, e_0, init_iter)

        step, _, _, _, _, _, _, _, _, _ = jax.lax.while_loop(wolfe_conds, update_alpha, init_args)

        x_temp = x_1 + step*p

        x_0 = x_1
        x_1 = x_temp

        D_0 = D_1
        gradD_0 = gradD_1

        B_0_inv = B_1_inv
        
        iter_num = iter_num + 1

    x_1 = jnp.reshape(x_1, (-1,))
        
    return x_1


@jax.jit
def find_min_bfgs(x, e_0) :

    return jax.vmap(BFGS, (0, 0))(x, e_0)

In [ ]:
class ClosestPoint(nn.Module):

    e_0: float
    
    @nn.compact
    def __call__(self, inputs):
    
        x_min = find_min_bfgs(inputs, self.e_0)
    
        return x_min

In [ ]:
class CDeepONet(nn.Module):

  t0: float
  tfinal: float
  layers: int
  units: int

  @nn.compact
  def __call__(self, t, u):

    if u.ndim == 1:
      u = jnp.reshape(u, (1,-1))

    if t.ndim == 1:
      t = jnp.reshape(t, (1,-1)) 

    batch_no = t.shape[0]

    t_0 = t[:, :1]
    H_0 = t[:, 1:2]
    eval1_0 = t[:, 2:3]
    eval2_0 = t[:, 3:4]
    eval3_0 = t[:, 4:]

    evals_new = t[:, 2:]
      
    t_0 = jnp.reshape(t_0, (-1,1))
    t_0_norm = Normalize(self.t0, self.tfinal)(t_0)
   
    p1_0 = u[:, :1]
    q1_0 = u[:, 1:2]
    p2_0 = u[:, 2:3]
    q2_0 = u[:, 3:4]
    p3_0 = u[:, 4:5]
    q3_0 = u[:, 5:]

    u = jnp.column_stack((p1_0, q1_0, p2_0, q2_0, p3_0, q3_0))

    # b = t_0_norm
    # b = jnp.column_stack((t_0_norm, H_0))
    b = jnp.column_stack((t_0_norm, eval1_0, eval2_0, eval3_0))

    trunk_net = MLP(self.layers, self.units)(b)
    branch_net = MLP(self.layers, self.units)(u)

    trunk_p1 = trunk_net[:, :self.units]
    trunk_q1 = trunk_net[:, self.units:2*self.units]
    trunk_p2 = trunk_net[:, 2*self.units:3*self.units]
    trunk_q2 = trunk_net[:, 3*self.units:4*self.units]
    trunk_p3 = trunk_net[:, 4*self.units:5*self.units]
    trunk_q3 = trunk_net[:, 5*self.units:]
      
    branch_p1 = branch_net[:, :self.units]
    branch_q1 = branch_net[:, self.units:2*self.units]
    branch_p2 = branch_net[:, 2*self.units:3*self.units]
    branch_q2 = branch_net[:, 3*self.units:4*self.units]
    branch_p3 = branch_net[:, 4*self.units:5*self.units]
    branch_q3 = branch_net[:, 5*self.units:]
      
    p1 = CombineBranches()(trunk_p1, branch_p1)
    q1 = CombineBranches()(trunk_q1, branch_q1)
    p2 = CombineBranches()(trunk_p2, branch_p2)
    q2 = CombineBranches()(trunk_q2, branch_q2)
    p3 = CombineBranches()(trunk_p3, branch_p3)
    q3 = CombineBranches()(trunk_q3, branch_q3)
      
    p1 = HardConstraint(self.t0, self.tfinal, p1_0)([t_0, p1])
    q1 = HardConstraint(self.t0, self.tfinal, q1_0)([t_0, q1])
    p2 = HardConstraint(self.t0, self.tfinal, p2_0)([t_0, p2])
    q2 = HardConstraint(self.t0, self.tfinal, q2_0)([t_0, q2])
    p3 = HardConstraint(self.t0, self.tfinal, p3_0)([t_0, p3])
    q3 = HardConstraint(self.t0, self.tfinal, q3_0)([t_0, q3])

    pq_arr = jnp.reshape(jnp.column_stack((p1, p2, p3, q1, q2, q3)), (batch_no, 6))
    # e_arr = jnp.reshape(jnp.column_stack((eval1_0, eval2_0, eval3_0)), (batch_no, 3))

    pq = ClosestPoint(evals_new)(pq_arr)

    p1, p2, p3, q1, q2, q3 = pq[:, :1], pq[:, 1:2], pq[:, 2:3], pq[:, 3:4], pq[:, 4:5], pq[:, 5:]
      
    p1 = jnp.reshape(p1, (-1,))
    q1 = jnp.reshape(q1, (-1,))
    p2 = jnp.reshape(p2, (-1,))
    q2 = jnp.reshape(q2, (-1,))
    p3 = jnp.reshape(p3, (-1,))
    q3 = jnp.reshape(q3, (-1,))

    # H = calc_H(p1, p2, p3, q1, q2, q3)
    # e = jax.vmap(calc_evals_pq, [0])((p1, p2, p3, q1, q2, q3))

    # evals = jax.vmap(calc_evals_pq, [0])((p1, p2, p3, q1, q2, q3))

    # eval1 = evals[:, :1]
    # eval2 = evals[:, 1:2]
    # eval3 = evals[:, 2:]

    # eval1 = jnp.reshape(eval1, (-1,))
    # eval2 = jnp.reshape(eval2, (-1,))
    # eval3 = jnp.reshape(eval3, (-1,))

    # print(eval1.shape)

    return p1, q1, p2, q2, p3, q3

In [ ]:
@partial(jax.jit, static_argnums=(2,))
def Cvar_model(t, z, component, params):
  return Cdeeponet.apply(params, t, z)[component][0]

@partial(jax.jit, static_argnums=(2,))
def Cvar(t, z, component, params):
  return jax.vmap(Cvar_model, [0, 0, None, None])(t, z, component, params)

@partial(jax.jit, static_argnums=(2,))
def Cvar_t(t, z, component, params):
  return jax.vmap(jax.grad(Cvar_model, 0), [0, 0, None, None])(t, z, component, params)       
    
@jax.jit
def Closs(params, t, z):
    
    p1_t_loss_w = 1.0
    q1_t_loss_w = 1.0
    p2_t_loss_w = 1.0
    q2_t_loss_w = 1.0
    p3_t_loss_w = 1.0
    q3_t_loss_w = 1.0    
    
    diff_loss_w = 1.0
    H_loss_w = 1.0
    # eval1_loss_w = 0.0
    # eval2_loss_w = 0.0
    # eval3_loss_w = 0.0

    t_0 = t[:, :1]
    H_0 = (t[:, 1])
    eval1_0 = t[:, 2]
    eval2_0 = t[:, 3]
    eval3_0 = t[:, 4]
    
    p1_0 = z[:, :1]
    q1_0 = z[:, 1:2]
    p2_0 = z[:, 2:3]
    q2_0 = z[:, 3:4]
    p3_0 = z[:, 4:5]
    q3_0 = z[:, 5:]

    p1 = Cvar(t, z, 0, params)
    q1 = Cvar(t, z, 1, params)
    p2 = Cvar(t, z, 2, params)
    q2 = Cvar(t, z, 3, params)
    p3 = Cvar(t, z, 4, params)
    q3 = Cvar(t, z, 5, params)
    # eval1 = Cvar(t, z, 6, params)
    # eval2 = Cvar(t, z, 7, params)
    # eval3 = Cvar(t, z, 8, params)
    
    
    H = calc_H(p1, p2, p3, q1, q2, q3)

    dp1_t = Cvar_t(t, z, 0, params)
    dq1_t = Cvar_t(t, z, 1, params)
    dp2_t = Cvar_t(t, z, 2, params)
    dq2_t = Cvar_t(t, z, 3, params)
    dp3_t = Cvar_t(t, z, 4, params)
    dq3_t = Cvar_t(t, z, 5, params)
    # deval1_t = Cvar_t(t, z, 6, params)
    # deval2_t = Cvar_t(t, z, 7, params)
    # deval3_t = Cvar_t(t, z, 8, params)
        
    p1_t = dp1_t[:, 0]
    q1_t = dq1_t[:, 0]  
    p2_t = dp2_t[:, 0]
    q2_t = dq2_t[:, 0]
    p3_t = dp3_t[:, 0]
    q3_t = dq3_t[:, 0]
    # eval1_t = deval1_t[:, 0]
    # eval2_t = deval2_t[:, 0]
    # eval3_t = deval3_t[:, 0]

    # evals_t_loss = jnp.mean(eval1_t**2) + jnp.mean(eval2_t**2) + jnp.mean(eval3_t**2)

    
    # evals = jax.vmap(calc_evals_pq, [0])((p1, p2, p3, q1, q2, q3))
    # eval1 = evals[:, 0]
    # eval2 = evals[:, 1]
    # eval3 = evals[:, 2]

    # print(jnp.max(jnp.abs(eval1_0**2 - eval1**2)))
    # print(jnp.max(jnp.abs(eval2_0**2 - eval2**2)))
    # print(jnp.max(jnp.abs(eval3_0**2 - eval3**2)))
    # print("H", jnp.max(jnp.abs(H_0 - H)))

    # print(H_0.shape, eval1_0.shape, eval2_0.shape, eval3_0.shape)
    # print(H_0)
    # print(2*(eval1_0**2 + eval2_0**2 + eval3_0**2))
    # input()
    
    H_loss = H - H_0
    H_loss = jnp.square(H_loss)
    H_loss = jnp.mean(H_loss)

    p1_t_loss = p1_t - (-jnp.exp(q1 - q2) + jnp.exp(q3 - q1))
    p1_t_loss = jnp.square(p1_t_loss)
    p1_t_loss = jnp.mean(p1_t_loss)

    q1_t_loss = q1_t - p1
    q1_t_loss = jnp.square(q1_t_loss)
    q1_t_loss = jnp.mean(q1_t_loss)
    
    p2_t_loss = p2_t - (jnp.exp(q1 - q2) - jnp.exp(q2 - q3))
    p2_t_loss = jnp.square(p2_t_loss)
    p2_t_loss = jnp.mean(p2_t_loss)
    
    q2_t_loss = q2_t - p2
    q2_t_loss = jnp.square(q2_t_loss)
    q2_t_loss = jnp.mean(q2_t_loss)

    p3_t_loss = p3_t - (jnp.exp(q2 - q3) - jnp.exp(q3 - q1))
    p3_t_loss = jnp.square(p3_t_loss)
    p3_t_loss = jnp.mean(p3_t_loss)
    
    q3_t_loss = q3_t - p3
    q3_t_loss = jnp.square(q3_t_loss)
    q3_t_loss = jnp.mean(q3_t_loss)
        
    diff_loss = p1_t_loss_w*p1_t_loss +  q1_t_loss_w*q1_t_loss + p2_t_loss_w*p2_t_loss + q2_t_loss_w*q2_t_loss + \
                p3_t_loss_w*p3_t_loss + q3_t_loss_w*q3_t_loss


    # evals_loss = jnp.mean((eval1 - eval1_0)**2) + jnp.mean((eval2 - eval2_0)**2) + jnp.mean((eval3 - eval3_0)**2)

    total_loss = diff_loss_w*diff_loss + H_loss

    
    return total_loss, H_loss
    
# @jax.jit
def Ctrain_step(params, pdes, z):
    t = pdes[:, :]

    test = jax.value_and_grad(Closs, has_aux = True)(params, t, z)
    
    return test

@partial(jax.jit, static_argnums=(3,))
def Coptimize(grads, opt_state, params, optimizer_update):
    updates, opt_state = optimizer_update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state

In [ ]:
def Ctrain_network(params, des, sensors, epochs=100):

    N = de_points.shape[0]
    
    nr_batches = 10
    batch_size = len(des)//nr_batches
    
    print("batch size:", batch_size)
    
    lr = 1e-4
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(params)
    
    ds_z = tf.data.Dataset.from_tensor_slices(sensors)
    ds_de = tf.data.Dataset.from_tensor_slices(des)
    
    ds = tf.data.Dataset.zip((ds_de, ds_z))
    ds = ds.shuffle(N).batch(batch_size)
    
    epoch_loss = np.zeros(epochs)
    epoch_aux_loss = np.zeros(epochs)

    for i in range(epochs):

        for (des_batch, z_batch) in ds:

            print("new batch")

            losses, grads = Ctrain_step(params, des_batch.numpy(), z_batch.numpy())
            print(losses)
            params, opt_state = Coptimize(grads, opt_state, params, optimizer.update)

            loss, aux_loss = losses
           
            epoch_loss[i] += loss
            epoch_aux_loss[i] += aux_loss

        epoch_loss[i] /= batch_size
        epoch_aux_loss[i] /= batch_size

        if i % 1  == 0:
            print(f'Loss in epoch {i}: {epoch_loss[i]}')
            print(f'Aux Loss in epoch {i}: {epoch_aux_loss[i]}')

    return params, epoch_loss

In [ ]:
Cdeeponet = CDeepONet(t0, tfinal, layers=4, units=40)

t_init = jnp.full((10, 5), 1.0, dtype = jnp.float64)
u_init = jnp.full((10, 6), 1.0, dtype = jnp.float64)

Cparams = Cdeeponet.init(jax.random.PRNGKey(0), t_init, u_init)

In [ ]:
Cparam_count = sum(x.size for x in jax.tree.leaves(Cparams))
print(Cparam_count)

In [ ]:
# epochs = 5
# Cparams, Closs = Ctrain_network(Cparams, de_points, zsensors, epochs)

# plt.semilogy(Closs)
# plt.grid()

In [ ]:
# tree_info, shapes, Cparams_flat = flatten_pytree(Cparams)
# jnp.save("C_t0.5_1000_50_ep_Toda3", Cparams_flat)

tree_info, shapes, _ = flatten_pytree(Cparams)
Cparams_flat = jnp.load("CHCons_0.5_0.5_t0.5_1000_100_ep_Toda3.npy")
Cparams = reconstruct_pytree(tree_info, shapes, Cparams_flat)

In [ ]:
@jax.jit
def loss_fun(args, e_0) :

    args = jnp.reshape(args, (-1,))
    e_0 = jnp.reshape(e_0, (-1,))

    p1, p2, p3, q1, q2, q3 = args[0], args[1], args[2], args[3], args[4], args[5]
    e1_0, e2_0, e3_0 = e_0[0], e_0[1], e_0[2]
    
    H_0 = 2*(e1_0**2 + e2_0**2 + e3_0**2)
    H = calc_H(p1, p2, p3, q1, q2, q3)

    f = (H - H_0)**2
       
    return jnp.array([[f]])

@jax.jit
def grad_loss_fun(args, e_0) :

    args = jnp.reshape(args, (-1,))
    e_0 = jnp.reshape(e_0, (-1,))

    p1, p2, p3, q1, q2, q3 = args[0], args[1], args[2], args[3], args[4], args[5]
    e1_0, e2_0, e3_0 = e_0[0], e_0[1], e_0[2]

    H_0 = 2*(e1_0**2 + e2_0**2 + e3_0**2)
    H = calc_H(p1, p2, p3, q1, q2, q3)

    dH_p1 = p1
    dH_p2 = p2
    dH_p3 = p3
    dH_q1 = jnp.exp(q1 - q2) - jnp.exp(q3 - q1)
    dH_q2 = -jnp.exp(q1 - q2) + jnp.exp(q2 - q3)
    dH_q3 = -jnp.exp(q2 - q3) + jnp.exp(q3 - q1)

    df_p1 = 2*(H - H_0)*dH_p1
    df_p2 = 2*(H - H_0)*dH_p2
    df_p3 = 2*(H - H_0)*dH_p3
    df_q1 = 2*(H - H_0)*dH_q1
    df_q2 = 2*(H - H_0)*dH_q2
    df_q3 = 2*(H - H_0)*dH_q3

    grad_f = jnp.array([[df_p1], [df_p2], [df_p3], [df_q1], [df_q2], [df_q3]])

    return grad_f

In [ ]:
@jax.jit
def find_rec(x):
    
    x = x + jnp.finfo(jnp.float64).eps
    
    return 1/x


@jax.jit
def wolfe_conds(args):

    alpha, curr_x, curr_f, curr_grad, next_x, next_f, next_grad, p, e_0, iter = args

    c1 = 1e-3

    cond = next_f - curr_f - c1*alpha*(curr_grad.T@p)

    cond_bool = jax.lax.cond(cond[0][0] > 0.0 , lambda x: True, lambda x: False, None)
    final_bool = cond_bool
    
    return final_bool


@jax.jit
def update_alpha(args) :

    alpha, curr_x, curr_f, curr_grad, next_x, next_f, next_grad, p, e_0, iter = args

    alpha = alpha*0.1
    iter = iter + 1
        
    next_x = curr_x + alpha*p
    next_f = loss_fun(next_x, e_0)
    next_grad = grad_loss_fun(next_x, e_0) 
   
    return (alpha, curr_x, curr_f, curr_grad, next_x, next_f, next_grad, p, e_0, iter)


@jax.jit
def BFGS(x, e_0) :

    iter_num = 0
    init_iter = 0
    alpha = 1.0
    I_3 = jnp.identity(6, dtype = jnp.float64)
    B_0_inv = jnp.identity(6, dtype = jnp.float64)

    e_0 = jnp.reshape(e_0, (-1, 1))
    
    x_0 = x
    x_0 = jnp.reshape(x_0, (-1, 1))

    D_0 = loss_fun(x_0, e_0)
    gradD_0 = grad_loss_fun(x_0, e_0)

    p = -(B_0_inv@gradD_0)

    pot_x = x_0 + alpha*p
    pot_D = loss_fun(pot_x, e_0)
    pot_gradD = grad_loss_fun(pot_x, e_0)

    init_args = (alpha, x_0, D_0, gradD_0, pot_x, pot_D, pot_gradD, p, e_0, init_iter)
    
    step, _, _, _, _, _, _, _, _, _ = jax.lax.while_loop(wolfe_conds, update_alpha, init_args)

    x_1 = x_0 + step*p

    while iter_num < 2:

        delta_x = x_1 - x_0

        D_1 = loss_fun(x_1, e_0)
        gradD_1 = grad_loss_fun(x_1, e_0)

        y = gradD_1 - gradD_0       

        y_T_delta_x = y.T@delta_x
        y_T_delta_x_rec = find_rec(y_T_delta_x)
        
        B_1_inv = ((I_3 - (y_T_delta_x_rec*(delta_x@y.T)))@B_0_inv@(I_3 - (y_T_delta_x_rec*(y@delta_x.T)))) + (y_T_delta_x_rec*(delta_x@delta_x.T))   
        p = -(B_1_inv@gradD_1)
        
        pot_x = x_1 + alpha*p
        pot_D = loss_fun(pot_x, e_0)
        pot_gradD = grad_loss_fun(pot_x, e_0) 
                  
        init_args = (alpha, x_1, D_1, gradD_1, pot_x, pot_D, pot_gradD, p, e_0, init_iter)

        step, _, _, _, _, _, _, _, _, _ = jax.lax.while_loop(wolfe_conds, update_alpha, init_args)

        x_temp = x_1 + step*p

        x_0 = x_1
        x_1 = x_temp

        D_0 = D_1
        gradD_0 = gradD_1

        B_0_inv = B_1_inv
        
        iter_num = iter_num + 1

    x_1 = jnp.reshape(x_1, (-1,))
        
    return x_1


@jax.jit
def find_min_bfgs(x, e_0) :

    return jax.vmap(BFGS, (0, 0))(x, e_0)

In [ ]:
class ClosestPointH(nn.Module):

    e_0: float
    
    @nn.compact
    def __call__(self, inputs):
    
        x_min = find_min_bfgs(inputs, self.e_0)
    
        return x_min

In [ ]:
class CHDeepONet(nn.Module):

  t0: float
  tfinal: float
  layers: int
  units: int

  @nn.compact
  def __call__(self, t, u):

    if u.ndim == 1:
      u = jnp.reshape(u, (1,-1))

    if t.ndim == 1:
      t = jnp.reshape(t, (1,-1)) 

    batch_no = t.shape[0]

    t_0 = t[:, :1]
    H_0 = t[:, 1:2]
    eval1_0 = t[:, 2:3]
    eval2_0 = t[:, 3:4]
    eval3_0 = t[:, 4:]

    evals_new = t[:, 2:]
      
    t_0 = jnp.reshape(t_0, (-1,1))
    t_0_norm = Normalize(self.t0, self.tfinal)(t_0)
   
    p1_0 = u[:, :1]
    q1_0 = u[:, 1:2]
    p2_0 = u[:, 2:3]
    q2_0 = u[:, 3:4]
    p3_0 = u[:, 4:5]
    q3_0 = u[:, 5:]

    u = jnp.column_stack((p1_0, q1_0, p2_0, q2_0, p3_0, q3_0))

    # b = t_0_norm
    # b = jnp.column_stack((t_0_norm, H_0))
    b = jnp.column_stack((t_0_norm, eval1_0, eval2_0, eval3_0))

    trunk_net = MLP(self.layers, self.units)(b)
    branch_net = MLP(self.layers, self.units)(u)

    trunk_p1 = trunk_net[:, :self.units]
    trunk_q1 = trunk_net[:, self.units:2*self.units]
    trunk_p2 = trunk_net[:, 2*self.units:3*self.units]
    trunk_q2 = trunk_net[:, 3*self.units:4*self.units]
    trunk_p3 = trunk_net[:, 4*self.units:5*self.units]
    trunk_q3 = trunk_net[:, 5*self.units:]
      
    branch_p1 = branch_net[:, :self.units]
    branch_q1 = branch_net[:, self.units:2*self.units]
    branch_p2 = branch_net[:, 2*self.units:3*self.units]
    branch_q2 = branch_net[:, 3*self.units:4*self.units]
    branch_p3 = branch_net[:, 4*self.units:5*self.units]
    branch_q3 = branch_net[:, 5*self.units:]
      
    p1 = CombineBranches()(trunk_p1, branch_p1)
    q1 = CombineBranches()(trunk_q1, branch_q1)
    p2 = CombineBranches()(trunk_p2, branch_p2)
    q2 = CombineBranches()(trunk_q2, branch_q2)
    p3 = CombineBranches()(trunk_p3, branch_p3)
    q3 = CombineBranches()(trunk_q3, branch_q3)
      
    p1 = HardConstraint(self.t0, self.tfinal, p1_0)([t_0, p1])
    q1 = HardConstraint(self.t0, self.tfinal, q1_0)([t_0, q1])
    p2 = HardConstraint(self.t0, self.tfinal, p2_0)([t_0, p2])
    q2 = HardConstraint(self.t0, self.tfinal, q2_0)([t_0, q2])
    p3 = HardConstraint(self.t0, self.tfinal, p3_0)([t_0, p3])
    q3 = HardConstraint(self.t0, self.tfinal, q3_0)([t_0, q3])

    pq_arr = jnp.reshape(jnp.column_stack((p1, p2, p3, q1, q2, q3)), (batch_no, 6))
    # e_arr = jnp.reshape(jnp.column_stack((eval1_0, eval2_0, eval3_0)), (batch_no, 3))

    pq = ClosestPointH(evals_new)(pq_arr)

    p1, p2, p3, q1, q2, q3 = pq[:, :1], pq[:, 1:2], pq[:, 2:3], pq[:, 3:4], pq[:, 4:5], pq[:, 5:]
      
    p1 = jnp.reshape(p1, (-1,))
    q1 = jnp.reshape(q1, (-1,))
    p2 = jnp.reshape(p2, (-1,))
    q2 = jnp.reshape(q2, (-1,))
    p3 = jnp.reshape(p3, (-1,))
    q3 = jnp.reshape(q3, (-1,))

    # H = calc_H(p1, p2, p3, q1, q2, q3)
    # e = jax.vmap(calc_evals_pq, [0])((p1, p2, p3, q1, q2, q3))

    # evals = jax.vmap(calc_evals_pq, [0])((p1, p2, p3, q1, q2, q3))

    # eval1 = evals[:, :1]
    # eval2 = evals[:, 1:2]
    # eval3 = evals[:, 2:]

    # eval1 = jnp.reshape(eval1, (-1,))
    # eval2 = jnp.reshape(eval2, (-1,))
    # eval3 = jnp.reshape(eval3, (-1,))

    # print(eval1.shape)

    return p1, q1, p2, q2, p3, q3

In [ ]:
@partial(jax.jit, static_argnums=(2,))
def CHvar_model(t, z, component, params):
  return CHdeeponet.apply(params, t, z)[component][0]

@partial(jax.jit, static_argnums=(2,))
def CHvar(t, z, component, params):
  return jax.vmap(CHvar_model, [0, 0, None, None])(t, z, component, params)

@partial(jax.jit, static_argnums=(2,))
def CHvar_t(t, z, component, params):
  return jax.vmap(jax.grad(CHvar_model, 0), [0, 0, None, None])(t, z, component, params)       
    
@jax.jit
def CHloss(params, t, z):
    
    p1_t_loss_w = 1.0
    q1_t_loss_w = 1.0
    p2_t_loss_w = 1.0
    q2_t_loss_w = 1.0
    p3_t_loss_w = 1.0
    q3_t_loss_w = 1.0    
    
    diff_loss_w = 1.0
    H_loss_w = 1.0
    # eval1_loss_w = 0.0
    # eval2_loss_w = 0.0
    # eval3_loss_w = 0.0

    t_0 = t[:, :1]
    H_0 = (t[:, 1])
    eval1_0 = t[:, 2]
    eval2_0 = t[:, 3]
    eval3_0 = t[:, 4]
    
    p1_0 = z[:, :1]
    q1_0 = z[:, 1:2]
    p2_0 = z[:, 2:3]
    q2_0 = z[:, 3:4]
    p3_0 = z[:, 4:5]
    q3_0 = z[:, 5:]

    p1 = CHvar(t, z, 0, params)
    q1 = CHvar(t, z, 1, params)
    p2 = CHvar(t, z, 2, params)
    q2 = CHvar(t, z, 3, params)
    p3 = CHvar(t, z, 4, params)
    q3 = CHvar(t, z, 5, params)
    # eval1 = CHvar(t, z, 6, params)
    # eval2 = CHvar(t, z, 7, params)
    # eval3 = CHvar(t, z, 8, params)
    
    
    H = calc_H(p1, p2, p3, q1, q2, q3)

    dp1_t = CHvar_t(t, z, 0, params)
    dq1_t = CHvar_t(t, z, 1, params)
    dp2_t = CHvar_t(t, z, 2, params)
    dq2_t = CHvar_t(t, z, 3, params)
    dp3_t = CHvar_t(t, z, 4, params)
    dq3_t = CHvar_t(t, z, 5, params)
    # deval1_t = CHvar_t(t, z, 6, params)
    # deval2_t = CHvar_t(t, z, 7, params)
    # deval3_t = CHvar_t(t, z, 8, params)
        
    p1_t = dp1_t[:, 0]
    q1_t = dq1_t[:, 0]  
    p2_t = dp2_t[:, 0]
    q2_t = dq2_t[:, 0]
    p3_t = dp3_t[:, 0]
    q3_t = dq3_t[:, 0]
    # eval1_t = deval1_t[:, 0]
    # eval2_t = deval2_t[:, 0]
    # eval3_t = deval3_t[:, 0]

    # evals_t_loss = jnp.mean(eval1_t**2) + jnp.mean(eval2_t**2) + jnp.mean(eval3_t**2)

    
    # evals = jax.vmap(calc_evals_pq, [0])((p1, p2, p3, q1, q2, q3))
    # eval1 = evals[:, 0]
    # eval2 = evals[:, 1]
    # eval3 = evals[:, 2]

    # print(jnp.max(jnp.abs(eval1_0**2 - eval1**2)))
    # print(jnp.max(jnp.abs(eval2_0**2 - eval2**2)))
    # print(jnp.max(jnp.abs(eval3_0**2 - eval3**2)))
    # print("H", jnp.max(jnp.abs(H_0 - H)))

    # print(H_0.shape, eval1_0.shape, eval2_0.shape, eval3_0.shape)
    # print(H_0)
    # print(2*(eval1_0**2 + eval2_0**2 + eval3_0**2))
    # input()
    
    H_loss = H - H_0
    H_loss = jnp.square(H_loss)
    H_loss = jnp.mean(H_loss)

    p1_t_loss = p1_t - (-jnp.exp(q1 - q2) + jnp.exp(q3 - q1))
    p1_t_loss = jnp.square(p1_t_loss)
    p1_t_loss = jnp.mean(p1_t_loss)

    q1_t_loss = q1_t - p1
    q1_t_loss = jnp.square(q1_t_loss)
    q1_t_loss = jnp.mean(q1_t_loss)
    
    p2_t_loss = p2_t - (jnp.exp(q1 - q2) - jnp.exp(q2 - q3))
    p2_t_loss = jnp.square(p2_t_loss)
    p2_t_loss = jnp.mean(p2_t_loss)
    
    q2_t_loss = q2_t - p2
    q2_t_loss = jnp.square(q2_t_loss)
    q2_t_loss = jnp.mean(q2_t_loss)

    p3_t_loss = p3_t - (jnp.exp(q2 - q3) - jnp.exp(q3 - q1))
    p3_t_loss = jnp.square(p3_t_loss)
    p3_t_loss = jnp.mean(p3_t_loss)
    
    q3_t_loss = q3_t - p3
    q3_t_loss = jnp.square(q3_t_loss)
    q3_t_loss = jnp.mean(q3_t_loss)
        
    diff_loss = p1_t_loss_w*p1_t_loss +  q1_t_loss_w*q1_t_loss + p2_t_loss_w*p2_t_loss + q2_t_loss_w*q2_t_loss + \
                p3_t_loss_w*p3_t_loss + q3_t_loss_w*q3_t_loss


    # evals_loss = jnp.mean((eval1 - eval1_0)**2) + jnp.mean((eval2 - eval2_0)**2) + jnp.mean((eval3 - eval3_0)**2)

    total_loss = diff_loss_w*diff_loss + H_loss

    
    return total_loss, H_loss
    
# @jax.jit
def CHtrain_step(params, pdes, z):
    t = pdes[:, :]

    test = jax.value_and_grad(CHloss, has_aux = True)(params, t, z)
    
    return test

@partial(jax.jit, static_argnums=(3,))
def CHoptimize(grads, opt_state, params, optimizer_update):
    updates, opt_state = optimizer_update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state

In [ ]:
def CHtrain_network(params, des, sensors, epochs=100):

    N = de_points.shape[0]
    
    nr_batches = 10
    batch_size = len(des)//nr_batches
    
    print("batch size:", batch_size)
    
    lr = 1e-4
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(params)
    
    ds_z = tf.data.Dataset.from_tensor_slices(sensors)
    ds_de = tf.data.Dataset.from_tensor_slices(des)
    
    ds = tf.data.Dataset.zip((ds_de, ds_z))
    ds = ds.shuffle(N).batch(batch_size)
    
    epoch_loss = np.zeros(epochs)
    epoch_aux_loss = np.zeros(epochs)

    for i in range(epochs):

        for (des_batch, z_batch) in ds:

            print("new batch")

            losses, grads = CHtrain_step(params, des_batch.numpy(), z_batch.numpy())
            print(losses)
            params, opt_state = CHoptimize(grads, opt_state, params, optimizer.update)

            loss, aux_loss = losses
           
            epoch_loss[i] += loss
            epoch_aux_loss[i] += aux_loss

        epoch_loss[i] /= batch_size
        epoch_aux_loss[i] /= batch_size

        if i % 1  == 0:
            print(f'Loss in epoch {i}: {epoch_loss[i]}')
            print(f'Aux Loss in epoch {i}: {epoch_aux_loss[i]}')

    return params, epoch_loss

In [ ]:
CHdeeponet = CHDeepONet(t0, tfinal, layers=4, units=40)

t_init = jnp.full((10, 5), 1.0, dtype = jnp.float64)
u_init = jnp.full((10, 6), 1.0, dtype = jnp.float64)

CHparams = CHdeeponet.init(jax.random.PRNGKey(0), t_init, u_init)

In [ ]:
CHparam_count = sum(x.size for x in jax.tree.leaves(CHparams))
print(CHparam_count)

In [ ]:
# epochs = 5
# CHparams, CHloss = CHtrain_network(CHparams, de_points, zsensors, epochs)

# plt.semilogy(CHloss)
# plt.grid()

In [ ]:
# tree_info, shapes, CHparams_flat = flatten_pytree(CHparams)
# jnp.save("CH_t0.5_1000_50_ep_Toda3", CHparams_flat)

tree_info, shapes, _ = flatten_pytree(CHparams)
CHparams_flat = jnp.load("CH_t0.5_1000_500_ep_Toda3.npy")
CHparams = reconstruct_pytree(tree_info, shapes, CHparams_flat)

In [ ]:
def run_test(num_models, num_exp, models_lst, params_lst, labels_lst, bfgs_lst) :

    n = 500
    t = np.linspace(t0, tfinal, n)
    
    
    preds = np.zeros( (num_exp, num_models, 7, n))
    preds_error = np.zeros((num_exp, num_models, 7))

    for i in range(num_exp) :

        print(i)

        p1_init = np.random.uniform(-1.0, 1.0, size=(1,))
        q1_init = np.random.uniform(-1.0, 1.0, size=(1,))
        p2_init = np.random.uniform(-1.0, 1.0, size=(1,))
        q2_init = np.random.uniform(-1.0, 1.0, size=(1,))
        p3_init = np.random.uniform(-1.0, 1.0, size=(1,))
        q3_init = np.random.uniform(-1.0, 1.0, size=(1,))
        H_init = calc_H(p1_init, p2_init, p3_init, q1_init, q2_init, q3_init)
        evals_init = calc_evals_pq((p1_init, p2_init, p3_init, q1_init, q2_init, q3_init))
        e1_init, e2_init, e3_init = evals_init[0], evals_init[1], evals_init[2]
        
        data_model = np.column_stack([p1_init, q1_init, p2_init, q2_init, p3_init, q3_init])
        data_model = np.reshape(data_model, (-1,))
        data_model = np.repeat(np.expand_dims(data_model, axis=0), n, axis=0)
        
        He_model = np.column_stack([H_init, e1_init, e2_init, e3_init])
        He_model = np.reshape(He_model, (-1,))
        He_model = np.repeat(np.expand_dims(He_model, axis=0), n, axis=0)
        
        t_model = np.column_stack([t, He_model])
        
        data_rk = np.column_stack([p1_init, q1_init, p2_init, q2_init, p3_init, q3_init])
        data_rk = np.reshape(data_rk, (-1,))
        t_rk = t
        
        sol_rk = odeint(TodaLatt, data_rk, t_rk, rtol = 1e-13, atol = 1e-13)
        p1_rk, q1_rk, p2_rk, q2_rk, p3_rk, q3_rk = sol_rk[:, 0], sol_rk[:, 1], sol_rk[:, 2], sol_rk[:, 3], sol_rk[:, 4], sol_rk[:, 5]
        H_rk = calc_H(p1_rk, p2_rk, p3_rk, q1_rk, q2_rk, q3_rk)
        H_data = np.repeat(np.expand_dims(H_init, axis=0), n, axis=0)
        tH_data = np.column_stack([t, H_data])

        for j in range(num_models) :

            model = models_lst[j]
            params = params_lst[j]

            for k in range(6) :

                val = jax.vmap(model, [0, 0, None, None])(t_model, data_model, k, params)
                preds[i, j, k] = val

            preds[i, j, 6] = calc_H(preds[i, j, 0], preds[i, j, 2], preds[i, j, 4], \
                                    preds[i, j, 1], preds[i, j, 3], preds[i, j, 5])

            if j in bfgs_lst :
                
                args_arr = np.column_stack([preds[i, j, 0], preds[i, j, 2], preds[i, j, 4], \
                                           preds[i, j, 1], preds[i, j, 3], preds[i, j, 5]])
                
                e_0_arr = t_model[:, 2:]
        
                xbfgs = find_min_bfgs(args_arr, e_0_arr)
                xbfgs_copy = np.copy(xbfgs)

                xbfgs = xbfgs.at[:, 1].set(xbfgs_copy[:, 3])
                xbfgs = xbfgs.at[:, 2].set(xbfgs_copy[:, 1])
                xbfgs = xbfgs.at[:, 3].set(xbfgs_copy[:, 4])
                xbfgs = xbfgs.at[:, 4].set(xbfgs_copy[:, 2])
                
                # xbfgs[:, 1] = xbfgs_copy[:, 3]
                # xbfgs[:, 2] = xbfgs_copy[:, 1]
                # xbfgs[:, 3] = xbfgs_copy[:, 4]
                # xbfgs[:, 4] = xbfgs_copy[:, 2]

                for l in range(6) :

                    preds[i, j, l] = xbfgs[:, l]
                    
                preds[i, j, 6] = calc_H(preds[i, j, 0], preds[i, j, 2], preds[i, j, 4], \
                                        preds[i, j, 1], preds[i, j, 3], preds[i, j, 5])
              

            for m in range(6) :

                preds_error[i, j, m] = np.mean(np.abs(sol_rk[:, m] - preds[i, j, m]))

            preds_error[i, j, 6] = np.mean(np.abs(H_rk - preds[i, j, 6]))


    fig = plt.figure(figsize=(25, 20))

    num = [number for number in range(num_exp)]
    x_labels = ["p1", "q1", "p2", "q2", "p3", "q3", "H"] 

    # for i in range(num_models) :

    for i in [3, 4] :
        
        for j in range(7) :
        
            plt.subplot(7, 1, j + 1)
            plt.tight_layout()
            #print(preds_error[:, i, j])
            plt.plot(num, preds_error[:, i, j], label = labels_lst[i])
            plt.xlabel(x_labels[j])
            plt.grid()
            plt.legend()

    plt.show()
    return preds, preds_error
        

In [ ]:
models_lst = [var_model, Evar_model, Evar_model, Cvar_model, CHvar_model]
params_lst = [params, Eparams, Eparams, Cparams, CHparams]
labels_lst = ["OG", "EX", "EXBFGS", "CONS", "CONSH"]
bfgs_lst = [2]

preds, preds_error = run_test(5, 50, models_lst, params_lst, labels_lst, bfgs_lst)

In [ ]:
def time_step_run_test(t_end_val, num_models, num_exp, models_lst, params_lst, labels_lst, bfgs_lst) :

    
    n = 500
    t_start = 0.0
    t_end = t_end_val

    n_ts = int(n*(t_end/tfinal))
    t_step = t_end/n_ts
    
    num_cycles = math.ceil(n_ts/n)

    t_ts = np.linspace(t_start, t_end, n_ts)
      
    preds = np.zeros((num_exp, num_cycles, num_models, 7, n))
    preds_error = np.zeros((num_exp, num_cycles, num_models, 7))
    
    for i in range(num_exp) :

        print(i)

        p1_init = np.random.uniform(-1.0, 1.0, size=(1,))
        q1_init = np.random.uniform(-1.0, 1.0, size=(1,))
        p2_init = np.random.uniform(-1.0, 1.0, size=(1,))
        q2_init = np.random.uniform(-1.0, 1.0, size=(1,))
        p3_init = np.random.uniform(-1.0, 1.0, size=(1,))
        q3_init = np.random.uniform(-1.0, 1.0, size=(1,))
        H_init = calc_H(p1_init, p2_init, p3_init, q1_init, q2_init, q3_init)
        evals_init = calc_evals_pq((p1_init, p2_init, p3_init, q1_init, q2_init, q3_init))
        e1_init, e2_init, e3_init = evals_init[0], evals_init[1], evals_init[2]

        data_rk = np.column_stack([p1_init, q1_init, p2_init, q2_init, p3_init, q3_init])
        data_rk = np.reshape(data_rk, (-1,))
        t_rk = t_ts
            
        sol_rk = odeint(TodaLatt, data_rk, t_rk, rtol = 1e-13, atol = 1e-13)
        p1_rk, q1_rk, p2_rk, q2_rk, p3_rk, q3_rk = sol_rk[:, 0], sol_rk[:, 1], sol_rk[:, 2], sol_rk[:, 3], sol_rk[:, 4], sol_rk[:, 5]
        H_rk = calc_H(p1_rk, p2_rk, p3_rk, q1_rk, q2_rk, q3_rk)
        
        for c in range(num_cycles) :
            
            for j in range(num_models) :

                if c != 0 :
    
                    p1_init = preds[i, c-1, j, 0, -1].reshape((1,))
                    q1_init = preds[i, c-1, j, 1, -1].reshape((1,))
                    p2_init = preds[i, c-1, j, 2, -1].reshape((1,))
                    q2_init = preds[i, c-1, j, 3, -1].reshape((1,))
                    p3_init = preds[i, c-1, j, 4, -1].reshape((1,))
                    q3_init = preds[i, c-1, j, 5, -1].reshape((1,))
                    H_init = calc_H(p1_init, p2_init, p3_init, q1_init, q2_init, q3_init)
                    evals_init = calc_evals_pq((p1_init, p2_init, p3_init, q1_init, q2_init, q3_init))
                    e1_init, e2_init, e3_init = evals_init[0], evals_init[1], evals_init[2]  

                data_model = np.column_stack([p1_init, q1_init, p2_init, q2_init, p3_init, q3_init])
                data_model = np.reshape(data_model, (-1,))
                data_model = np.repeat(np.expand_dims(data_model, axis=0), n, axis=0)
                
                He_model = np.column_stack([H_init, e1_init, e2_init, e3_init])
                He_model = np.reshape(He_model, (-1,))
                He_model = np.repeat(np.expand_dims(He_model, axis=0), n, axis=0)
                
                t_model = np.column_stack([t, He_model])

                
                model = models_lst[j]
                params = params_lst[j]
    
                for k in range(6) :
    
                    val = jax.vmap(model, [0, 0, None, None])(t_model, data_model, k, params)
                    preds[i, c, j, k] = val
    
                preds[i, c, j, 6] = calc_H(preds[i, c, j, 0], preds[i, c, j, 2], preds[i, c, j, 4], \
                                        preds[i, c, j, 1], preds[i, c, j, 3], preds[i, c, j, 5])
    
                if j in bfgs_lst :
                    
                    args_arr = np.column_stack([preds[i, c, j, 0], preds[i, c, j, 2], preds[i, c, j, 4], \
                                               preds[i, c, j, 1], preds[i, c, j, 3], preds[i, c, j, 5]])
                    
                    e_0_arr = t_model[:, 2:]
            
                    xbfgs = find_min_bfgs(args_arr, e_0_arr)
                    xbfgs_copy = np.copy(xbfgs)
    
                    xbfgs = xbfgs.at[:, 1].set(xbfgs_copy[:, 3])
                    xbfgs = xbfgs.at[:, 2].set(xbfgs_copy[:, 1])
                    xbfgs = xbfgs.at[:, 3].set(xbfgs_copy[:, 4])
                    xbfgs = xbfgs.at[:, 4].set(xbfgs_copy[:, 2])

                    
                    for l in range(6) :
    
                        preds[i, c, j, l] = xbfgs[:, l]
                        
                    preds[i, c, j, 6] = calc_H(preds[i, c, j, 0], preds[i, c, j, 2], preds[i, c, j, 4], \
                                            preds[i, c, j, 1], preds[i, c, j, 3], preds[i, c, j, 5])
                  
    
                for m in range(6) :
    
                    preds_error[i, c, j, m] = np.mean(np.abs(sol_rk[c*n:(c+1)*n, m] - preds[i, c, j, m]))
    
                preds_error[i, c, j, 6] = np.mean(np.abs(H_rk[c*n:(c+1)*n] - preds[i, c, j, 6]))
    

    fig = plt.figure(figsize=(25, 20))

    num = [number for number in range(num_exp)]
    x_labels = ["p1", "q1", "p2", "q2", "p3", "q3", "H"] 


    for i in [0, 3, 4] :

        for j in range(7) :
            
            plt.subplot(7, 1, j + 1)
            plt.plot(t_ts, preds[0, :, i, j].flatten(), label = labels_lst[i])
                
            plt.xlabel(x_labels[j])
            plt.grid()
            plt.legend()

    for j in range(7) :
    
        plt.subplot(7, 1, j + 1)
        
        if j != 6 :
            plt.plot(t_ts, sol_rk[:, j], label = "RK")
        else:
            plt.plot(t_ts, H_rk, label = "RK")
            
        plt.xlabel(x_labels[j])
        plt.grid()
        plt.legend()
    
    plt.show()
    
    return preds, preds_error


    

In [ ]:
preds, preds_error = time_step_run_test(15.0, 5, 1, models_lst, params_lst, labels_lst, bfgs_lst)

In [ ]:
def compare_methods(num_trials, t_end_val) :

    n = 500
    trials_num = [i+1 for i in range(num_trials)]
    
    t_start = 0.0
    t_end = t_end_val
    
    n_ts = int(n*(t_end/tfinal))
    
    t_step = t_end/n_ts
    
    num_cycles = math.ceil(n_ts/n)
    
    t_ts = np.linspace(t_start, t_end, n_ts)
 
    A_errors_bfgs, F_errors_bfgs, G_errors_bfgs  = [], [], []
    A_errors_lbfgs, F_errors_lbfgs, G_errors_lbfgs = [], [], []
    
    for i in range(num_trials) :

        print("Trial no", i + 1)

        A_init = np.random.uniform(-1.0, 1.0, size=(1,))
        F_init = np.random.uniform(-1.0, 1.0, size=(1,))
        G_init = np.random.uniform(-1.0, 1.0, size=(1,))
        E_init = (1/4)*((A_init**2/l**2) + (F_init**2/k**2) + (2*(G_init**2)/(k**2 + l**2)))
        V_init = (1/2)*(A_init**2 + F_init**2 + 2*(G_init**2))
        
        # A_init, F_init, G_init, E_init, V_init = set_vals([[-0.24342532], [0.31170817], [-0.85430722]])
        
        data_rk = np.column_stack([A_init, F_init, G_init])
        data_rk = np.reshape(data_rk, (-1,))
        t_rk = t_ts
        
        sol_rk = odeint(Lorenz1960, data_rk, t_rk, rtol = 1e-13, atol = 1e-13)
        A_rk, F_rk, G_rk = sol_rk[:,0], sol_rk[:,1], sol_rk[:,2]
        E_rk = (1/4)*((A_rk**2/l**2) + (F_rk**2/k**2) + (2*(G_rk**2)/(k**2 + l**2)))
        V_rk = (1/2)*(A_rk**2 + F_rk**2 + 2*(G_rk**2))
        
        A_pred_ovan, F_pred_ovan, G_pred_ovan, E_pred_ovan, V_pred_ovan = init_preds(n_ts)
        
        A_pred_van, F_pred_van, G_pred_van, E_pred_van, V_pred_van = init_preds(n_ts)
        
        A_pred_bfgs, F_pred_bfgs, G_pred_bfgs, E_pred_bfgs, V_pred_bfgs = init_preds(n_ts)
        
        A_pred_lbfgs, F_pred_lbfgs, G_pred_lbfgs, E_pred_lbfgs, V_pred_lbfgs = init_preds(n_ts)
        
        A_data_ovan, A_data_van, A_data_bfgs, A_data_lbfgs = A_init, A_init, A_init, A_init
        F_data_ovan, F_data_van, F_data_bfgs, F_data_lbfgs = F_init, F_init, F_init, F_init
        G_data_ovan, G_data_van, G_data_bfgs, G_data_lbfgs = G_init, G_init, G_init, G_init
        E_data_ovan, E_data_van, E_data_bfgs, E_data_lbfgs = E_init, E_init, E_init, E_init
        V_data_ovan, V_data_van, V_data_bfgs, V_data_lbfgs = V_init, V_init, V_init, V_init
        
        
        t_data = np.linspace(t0, tfinal, n)
        t_data_sub = np.linspace(t0 + t_step, tfinal, n)
        
        for i in range(num_cycles) :
        
            if i != 0 :
                t_data = t_data_sub
        
            A_ovan, F_ovan, G_ovan, E_ovan, V_ovan, _, _ = prep_train(t_data, 
                                                           A_data_ovan, F_data_ovan, G_data_ovan, E_data_ovan, V_data_ovan, 
                                                           Vvar_model, Vparams, n)    
        
            A_van, F_van, G_van, E_van, V_van, _, _ = prep_train(t_data, 
                                                           A_data_van, F_data_van, G_data_van, E_data_van, V_data_van, 
                                                           var_model, params, n)
        
            A_bfgs, F_bfgs, G_bfgs, E_bfgs, V_bfgs, _, _ = prep_train(t_data, 
                                                           A_data_bfgs, F_data_bfgs, G_data_bfgs, E_data_bfgs, V_data_bfgs, 
                                                           Cvar_model, params, n)  
            
            A_lbfgs, F_lbfgs, G_lbfgs, E_lbfgs, V_lbfgs, _, _ = prep_train(t_data, 
                                                           A_data_lbfgs, F_data_lbfgs, G_data_lbfgs, E_data_lbfgs, V_data_lbfgs, 
                                                           Cvar_model, Cparams, n)    
        
            A_pred_ovan[i*n:(i+1)*n] = A_ovan
            F_pred_ovan[i*n:(i+1)*n] = F_ovan
            G_pred_ovan[i*n:(i+1)*n] = G_ovan
            E_pred_ovan[i*n:(i+1)*n] = E_ovan
            V_pred_ovan[i*n:(i+1)*n] = V_ovan
        
            A_pred_van[i*n:(i+1)*n] = A_van
            F_pred_van[i*n:(i+1)*n] = F_van
            G_pred_van[i*n:(i+1)*n] = G_van
            E_pred_van[i*n:(i+1)*n] = E_van
            V_pred_van[i*n:(i+1)*n] = V_van
            
            A_pred_bfgs[i*n:(i+1)*n] = A_bfgs
            F_pred_bfgs[i*n:(i+1)*n] = F_bfgs
            G_pred_bfgs[i*n:(i+1)*n] = G_bfgs
            E_pred_bfgs[i*n:(i+1)*n] = E_bfgs
            V_pred_bfgs[i*n:(i+1)*n] = V_bfgs
            
            A_pred_lbfgs[i*n:(i+1)*n] = A_lbfgs
            F_pred_lbfgs[i*n:(i+1)*n] = F_lbfgs
            G_pred_lbfgs[i*n:(i+1)*n] = G_lbfgs
            E_pred_lbfgs[i*n:(i+1)*n] = E_lbfgs
            V_pred_lbfgs[i*n:(i+1)*n] = V_lbfgs
        
            A_data_ovan = A_pred_ovan[(i+1)*n - 1]
            F_data_ovan = F_pred_ovan[(i+1)*n - 1]
            G_data_ovan = G_pred_ovan[(i+1)*n - 1]
            E_data_ovan = E_pred_ovan[(i+1)*n - 1]
            V_data_ovan = V_pred_ovan[(i+1)*n - 1]
         
            A_data_van = A_pred_van[(i+1)*n - 1]
            F_data_van = F_pred_van[(i+1)*n - 1]
            G_data_van = G_pred_van[(i+1)*n - 1]
            E_data_van = E_pred_van[(i+1)*n - 1]
            V_data_van = V_pred_van[(i+1)*n - 1]
        
            A_data_bfgs = A_pred_bfgs[(i+1)*n - 1]
            F_data_bfgs = F_pred_bfgs[(i+1)*n - 1]
            G_data_bfgs = G_pred_bfgs[(i+1)*n - 1]
            E_data_bfgs = E_pred_bfgs[(i+1)*n - 1]
            V_data_bfgs = V_pred_bfgs[(i+1)*n - 1]
            
            A_data_lbfgs = A_pred_lbfgs[(i+1)*n - 1]
            F_data_lbfgs = F_pred_lbfgs[(i+1)*n - 1]
            G_data_lbfgs = G_pred_lbfgs[(i+1)*n - 1]
            E_data_lbfgs = E_pred_lbfgs[(i+1)*n - 1]
            V_data_lbfgs = V_pred_lbfgs[(i+1)*n - 1]

        # fig = plt.figure(figsize=(8,8))
        # ax = fig.add_subplot(projection='3d')
        # #ax.plot(A_pred_ovan, F_pred_ovan, G_pred_ovan, label='OVanilla')
        # #ax.plot(A_pred_van, F_pred_van, G_pred_van, label='Vanilla')
        # ax.plot(A_pred_bfgs, F_pred_bfgs, G_pred_bfgs, label='BFGS')
        # ax.plot(A_pred_lbfgs, F_pred_lbfgs, G_pred_lbfgs, label='LBFGS')
        # ax.plot(A_rk, F_rk, G_rk, linestyle='--', label='RK45')
        # ax.legend()
        # ax.set_xlabel('A')
        # ax.set_ylabel('F')
        # ax.set_zlabel('G')
        # plt.show()
        
        A_err_bfgs = np.mean(np.abs(A_pred_bfgs - A_rk))
        F_err_bfgs = np.mean(np.abs(F_pred_bfgs - F_rk))
        G_err_bfgs = np.mean(np.abs(G_pred_bfgs - G_rk))

        A_err_lbfgs = np.mean(np.abs(A_pred_lbfgs - A_rk))
        F_err_lbfgs = np.mean(np.abs(F_pred_lbfgs - F_rk))
        G_err_lbfgs = np.mean(np.abs(G_pred_lbfgs - G_rk))

        A_errors_bfgs.append(A_err_bfgs)
        F_errors_bfgs.append(F_err_bfgs)
        G_errors_bfgs.append(G_err_bfgs)

        A_errors_lbfgs.append(A_err_lbfgs)
        F_errors_lbfgs.append(F_err_lbfgs)
        G_errors_lbfgs.append(G_err_lbfgs)


    fig = plt.figure(figsize=(12, 9))
    
    plt.subplot(3, 1, 1)
    plt.tight_layout()
    plt.plot(trials_num, A_errors_bfgs, label = 'BFGS')
    plt.plot(trials_num, A_errors_lbfgs, label = 'LBFGS')
    plt.xlabel("Trial no")
    plt.ylabel("MAE in A")
    plt.xticks(trials_num)
    plt.grid()
    plt.legend()
    
    plt.subplot(3, 1, 2)
    plt.tight_layout()
    plt.plot(trials_num, F_errors_bfgs, label = 'BFGS')
    plt.plot(trials_num, F_errors_lbfgs, label = 'LBFGS')
    plt.xlabel("Trial no")
    plt.ylabel("MAE in F")
    plt.xticks(trials_num)
    plt.grid()
    plt.legend()
    
    plt.subplot(3, 1, 3)
    plt.tight_layout()
    plt.plot(trials_num, G_errors_bfgs, label = 'BFGS')
    plt.plot(trials_num, G_errors_lbfgs, label = 'LBFGS')
    plt.xlabel("Trial no")
    plt.ylabel("MAE in G")
    plt.xticks(trials_num)
    plt.grid()
    plt.legend()
    
    plt.show()
    
    return 